# 3 · Claude Code II: Debugging, Testing & the Earnings Engine

**Outcome of this session:** an AI-generated investment-memo draft from your own Earnings Analysis Engine, with a verification layer that detects fabricated evidence automatically.

## The debugging protocol

1. **Read the traceback bottom-up**: the last line says *what* happened, the marked line says *where*.
2. **Reproduce the error** before changing anything.
3. **Diagnose before fixing**: make Claude explain the cause first.
4. **Change one thing at a time.**

> A crash announces itself and forces a fix. The dangerous error is the one that returns a plausible but wrong number.

**Tests are financial logic written down.** The most important one is the *hand-check*: a test case simple enough to compute by hand, so the correct answer is known before the code runs. A model that cannot reproduce a hand-checkable case is not yet a model.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A: the broken DCF

`session-03-debugging/demo/broken_dcf.py` contains a **discounted cash flow (DCF) valuation** of Meridian Semiconductor, a fictional company trading at $62. A DCF estimates what a company is worth today by projecting its future free cash flows and discounting them back to the present. This copy ships with six planted bugs. Run it and read the failure:

In [ ]:
import subprocess
print("EXPECTED FAILURE - broken_dcf.py is deliberately broken; you will fix it in the lab.")
print("=" * 78)
r = subprocess.run([sys.executable, str(ROOT / "session-03-debugging" / "demo" / "broken_dcf.py")],
                   capture_output=True, text=True)
print(r.stdout[-300:] if r.stdout else "", r.stderr[-500:])
print("=" * 78)
print("Diagnosis, bottom-up: WHAT = the last line (IndexError: the growth-rate list")
print("ran out). WHERE = the marked line (project_fcf, line 42: the loop assumes")
print("more years than the list contains - a hardcoded horizon). This is planted")
print("bug 1 of 6, and the only one that announces itself.")

In [ ]:
# and watch its sanity tests fail (7 failures = 7 pieces of violated financial logic)
r = subprocess.run([sys.executable, "-m", "pytest", str(ROOT / "session-03-debugging" / "demo"), "-q"],
                   capture_output=True, text=True)
print(r.stdout[-600:])

You have seen the demonstration: the crash diagnosed, one function repaired with the ✱ panel, and the partially fixed model still printing $115 for a $62 stock. The remaining errors are yours to remove — not by patching the broken file, but by **building the correct model yourself**, function by function, each with its test directly below. The specifications are in the docstrings; the finance is in the comments.

## Part B · Lab 1: build the DCF correctly

### Exercise 1: WACC

The **WACC**, or weighted average cost of capital, is the DCF's discount rate: the cost of equity and the cost of debt, each weighted by its share of the financing. Include the **tax shield**: interest is tax-deductible, so the effective cost of debt is reduced by a factor of (1 − tax rate).

In [ ]:
def wacc(equity_weight, debt_weight, cost_of_equity, cost_of_debt, tax_rate):
    """WACC = w_e * k_e + w_d * k_d * (1 - tax_rate). Interest is tax-deductible:
    debt is cheaper than it looks."""
### START CODE HERE ###
    return equity_weight * None + debt_weight * None * (1 - None)   # don't forget the tax shield
### END CODE HERE ###

print(f"Meridian WACC: {wacc(0.85, 0.15, 0.115, 0.055, 0.21):.2%}")

In [ ]:
# ✅ self-check: run me
assert abs(wacc(0.0, 1.0, 0.10, 0.05, 0.21) - 0.05 * 0.79) < 1e-12, "100% debt at 5%, 21% tax -> 3.95%"
assert abs(wacc(1.0, 0.0, 0.115, 0.05, 0.21) - 0.115) < 1e-12
print("All checks passed ✅")

### Exercise 2: project cash flows

Project the yearly **free cash flow (FCF)**, the cash the business generates after its investment needs: starting from the base year, each year grows the previous one by that year's growth rate. The function must accept a growth-rate list of any length; the horizon is however many rates it receives.

In [ ]:
def project_fcf(base_fcf, growth_rates):
    """project_fcf(100, [0.10, 0.10]) -> [110.0, 121.0]. Horizon = len(growth_rates),
    never hardcoded."""
### START CODE HERE ###
    flows, fcf = [], base_fcf
    for g in growth_rates:
        fcf = fcf * (1 + None)
        flows.append(None)
    return flows
### END CODE HERE ###

project_fcf(1.35, [0.30, 0.25, 0.20, 0.15, 0.10])

In [ ]:
# ✅ self-check: run me
f = project_fcf(100.0, [0.10, 0.10, 0.10])
assert len(f) == 3, "3 rates in -> 3 flows out (the broken model hardcoded 5 and crashed)"
assert abs(f[2] - 133.1) < 1e-9
print("All checks passed ✅")

### Exercise 3: terminal value

The **terminal value (TV)** captures all value beyond the explicit forecast years, as a growing perpetuity: `TV = FCF_final × (1 + g) / (r − g)`, where `g` is the perpetual growth rate and `r` the discount rate. If `g >= r` the formula implies infinite value, which is a sign of impossible inputs rather than an attractive company: **raise `ValueError`** instead of dividing. Validation is part of the model.

In [ ]:
def terminal_value(final_fcf, terminal_growth, discount_rate):
### START CODE HERE ###
    if None >= None:                       # which comparison implies infinite value?
        raise ValueError("terminal growth must be below the discount rate - otherwise value is infinite")
    return final_fcf * (1 + None) / (None - None)
### END CODE HERE ###

print(f"TV example: {terminal_value(100, 0.02, 0.10):,.0f}")

In [ ]:
# ✅ self-check: run me
assert abs(terminal_value(100, 0.0, 0.10) - 1000.0) < 1e-9
try:
    terminal_value(100, 0.12, 0.10)
    raise AssertionError("g >= r must raise ValueError, not return a number")
except ValueError:
    pass
print("All checks passed ✅")

### Exercise 4: the full DCF

Discount year *t* at `(1 + r)**t`, with **t = 1 for the first year**: a cash flow one year away is worth less than the same amount today. The terminal value sits at the end of year N, so discount it by `(1 + r)**N`. Equity value is enterprise value **minus** net debt, because debt holders are paid first.

In [ ]:
def dcf_value(base_fcf, growth_rates, discount_rate, terminal_growth, net_debt, shares_outstanding):
### START CODE HERE ###
    flows = project_fcf(base_fcf, growth_rates)
    n = len(flows)
    pv_explicit = sum(f / (1 + discount_rate) ** t
                      for t, f in enumerate(flows, start=None))   # the FIRST year is t = ?
    pv_terminal = terminal_value(flows[-1], terminal_growth, discount_rate) / (1 + discount_rate) ** None
    enterprise_value = None + None
    equity_value = enterprise_value - None                        # who gets paid first?
    per_share = equity_value / None
### END CODE HERE ###
    return {"enterprise_value": enterprise_value, "equity_value": equity_value,
            "per_share": per_share, "pv_explicit": pv_explicit, "pv_terminal": pv_terminal}

In [ ]:
# ✅ self-check: the NAPKIN TEST: flat FCF 100, r=10%, g=0, 2 years.
# PV explicit = 100/1.1 + 100/1.21 = 173.55; TV = 1000, PV(TV) = 826.45; EV = 1000.00 exactly.
r = dcf_value(100.0, [0.0, 0.0], 0.10, 0.0, net_debt=200.0, shares_outstanding=10.0)
assert abs(r["enterprise_value"] - 1000.0) < 0.01, f"EV should be 1000.00, got {r['enterprise_value']:.2f}"
assert abs(r["equity_value"] - 800.0) < 0.01, "equity = EV - net debt (SUBTRACT)"
assert abs(r["per_share"] - 80.0) < 0.001
print("All checks passed ✅ - your model reproduces a hand-checkable case")

In [ ]:
# Now value Meridian for real:
rate = wacc(0.85, 0.15, 0.115, 0.055, 0.21)
result = dcf_value(1.35, [0.30, 0.25, 0.20, 0.15, 0.10], rate, 0.025,
                   net_debt=0.85, shares_outstanding=0.46)
print(f"WACC {rate:.2%} | EV ${result['enterprise_value']:.2f}bn | "
      f"per share ${result['per_share']:.2f}  (market: $62.00)")
assert abs(result["per_share"] - 75.61) < 0.05, "expected ~$75.61/share"
print("+22% upside - IF you believe the growth assumptions. Green tests AND a plausible number: you need both.")

## Part C · Lab 2: the Earnings Analysis Engine

From an earnings-call transcript to a structured, **evidence-verified** note. The transcript (`session-03-debugging/data/transcript_meridian_q2_fy2026.txt`) is synthetic and the company fictional, so the model cannot rely on memorized knowledge; every claim must come from the document.

The plumbing (schema + API call) is imported from the course solution; **your work is the trust layer**.

In [ ]:
sol_dir = ROOT / "session-03-debugging" / "solutions"
sys.path.insert(0, str(sol_dir))
from earnings_engine import EARNINGS_SCHEMA, analyze, render_memo, DEFAULT_TRANSCRIPT

transcript = DEFAULT_TRANSCRIPT.read_text()
print(f"{len(transcript.split())} words. Speakers: CEO, CFO, five analysts. Somewhere in here: 7 red flags.")

### Exercise 5: verify_evidence, the fabrication detector

Every claim the model makes carries a verbatim quote. You check each quote against the source **in plain Python**. Normalize both sides first (collapse whitespace, lowercase, straighten curly quotes) so that formatting differences cannot cause false negatives. Set `item["verified"]` on every item in `key_themes`, `risks`, `red_flags`, and store totals in `analysis["_verification"]`.

In [ ]:
import re

def _normalize(text: str) -> str:
    """Whitespace-collapse + casefold + straighten curly quotes. GIVEN - it's
    plumbing; YOUR work is the verification logic below."""
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    return re.sub(r"\s+", " ", text).casefold().strip()

def verify_evidence(analysis: dict, transcript: str) -> dict:
### START CODE HERE ###
    haystack = _normalize(None)                        # normalize which text?
    checked = failed = 0
    for section in ("key_themes", "risks", "red_flags"):
        for item in analysis.get(section, []):
            quote = item.get("evidence_quote", "")
            item["verified"] = bool(quote) and None in haystack   # hint: the NORMALIZED quote
            checked += 1
            failed += 0 if item["verified"] else 1
    analysis["_verification"] = {"quotes_checked": None, "quotes_failed": None}
### END CODE HERE ###
    return analysis

print("defined - now catch a fabrication:")

In [ ]:
# ✅ self-check: run me. The canned dry-run analysis hides ONE deliberately
# fabricated quote. If your verify_evidence works, it catches exactly that one.
analysis = verify_evidence(analyze(transcript, dry_run=True), transcript)
v = analysis["_verification"]
print(f"quotes checked: {v['quotes_checked']}, failed: {v['quotes_failed']}")
assert v["quotes_checked"] >= 10, "check key_themes, risks AND red_flags"
assert v["quotes_failed"] == 1, "exactly ONE quote is fabricated - if 0, your matching is too loose; if >1, normalize better"
fake = [t for s in ("key_themes", "risks", "red_flags") for t in analysis[s] if not t["verified"]]
print("All checks passed ✅  Caught fabrication:", repr(fake[0]["evidence_quote"]))

That quote, a promised margin recovery, is entirely plausible and appears nowhere in the transcript. **Reading alone would likely have missed it; your code did not.** This is verification in practice.

**What to expect on a live run.** The cell below also runs the engine against the real API if you have a key. There the model writes its own quotes, and a current model usually quotes accurately, so you should expect **0 failures**. That is your checker working, not failing: it reports what it finds. The planted fabrication exists in the canned analysis precisely so that every student sees a catch at least once. In production the value of this layer is not that it fires often; it is that nothing reaches a committee unchecked.

In [ ]:
# Render the full memo (uses the course renderer) and, if you have a key, run LIVE:
OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)
(OUTD / "meridian_earnings_memo.md").write_text(render_memo(analysis, "transcript (dry-run)"))
print("outputs/meridian_earnings_memo.md written (dry-run).")

if HAS_KEY:
    live = verify_evidence(analyze(transcript, dry_run=False), transcript)
    lv = live["_verification"]
    (OUTD / "meridian_earnings_memo.md").write_text(render_memo(live, "transcript (LIVE)"))
    verified = lv["quotes_checked"] - lv["quotes_failed"]
    print(f"LIVE run: {verified}/{lv['quotes_checked']} quotes verified against the transcript.")
    print("0 failures is the expected result with a current model: the checker reports, it does not accuse.")
    print()
    print("Now read YOUR memo. Did it find: recurring 'one-time' costs? the CEO/CFO margin")
    print("gap? the guidance exclusion on the export review? DSO at 71 days versus 58?")
else:
    print("No API key - the dry-run memo still demonstrates the whole pipeline.")

## Deliverable checklist

- [ ] All DCF checks green, ending at **$75.61 vs $62 market**, and you can name each planted bug in one sentence:
  - the crash
  - year-1 discounting
  - the tax shield
  - the undiscounted terminal value
  - the net-debt sign
  - the missing g < r guard
- [ ] Your `verify_evidence` catches **exactly 1** fabricated quote in dry-run
- [ ] `outputs/meridian_earnings_memo.md` generated (live if you have a key) and committed to your repo
- [ ] Stretch: numeric cross-check: regex the transcript for figures and confirm every number in the summary appears in the source

**Next:** `04-workflows-edgar.ipynb`, where we stop pasting context by hand and start fetching it programmatically, from live SEC filings.